<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_11/Day_1%20/Daily_Challenge_Inventory_Analysis_in_Tableau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge: Inventory Analysis in Tableau
===


**Your Task**


- Utilize the provided ‘Acropolis Retail Data’ dataset: case-study-inventory-analysis-in-tableau.

- Data Preparation: Load the dataset into Tableau and prepare it for analysis, ensuring data quality and relevance.

- Calculated Fields: Create calculated fields in Tableau to analyze inventory management metrics, including inventory turnover rates.
- ABC Analysis: Conduct ABC analysis to categorize products based on their importance to inventory management.
- Visualization: Develop a set of visualizations that effectively communicate your findings from the inventory turnover and ABC analysis.
- Dashboard Creation: Assemble your visualizations into a single, interactive dashboard tailored for inventory analysis.

# WarmeHands Inc. - Inventory Analysis & ABC Classification

This notebook processes inventory and sales data for **WarmeHands Inc.**, calculates key inventory management metrics, performs ABC analysis, and displays an interactive Tableau dashboard.

In [1]:
import pandas as pd

# Load Excel Sheets
stock_df = pd.read_excel('Acropolis_Retail_data.xlsx', sheet_name='Stock')
orders_df = pd.read_excel('Acropolis_Retail_data.xlsx', sheet_name='Orders')

# Aggregate 2021 Sales
sku_sales = orders_df.groupby('SKU-ID')['Quantity'].sum().reset_index()
sku_sales.rename(columns={'Quantity': '2021_Units_Sold'}, inplace=True)

# Merge datasets
df = pd.merge(stock_df, sku_sales, on='SKU-ID', how='left').fillna({'2021_Units_Sold': 0})

# Calculated Metrics
df['Unit_COGS'] = (
    df['Raw_Material_Cost'] +
    df['Factory_Labor_Costs'] +
    df['Factory_Equipment_Rent_Costs'] +
    df['Distribution_Costs'] +
    df['Advertisement_Costs']
)
df['Total_Sales_Revenue'] = df['2021_Units_Sold'] * df['Retail_Price']
df['Total_COGS'] = df['2021_Units_Sold'] * df['Unit_COGS']
df['Avg_Inventory_Value'] = df['2021_start_stock'] * df['Unit_COGS']

# ABC Analysis Classification
df = df.sort_values(by='Total_Sales_Revenue', ascending=False).reset_index(drop=True)
df['Cum_Revenue'] = df['Total_Sales_Revenue'].cumsum()
df['Cum_Pct'] = df['Cum_Revenue'] / df['Total_Sales_Revenue'].sum()

def assign_abc(pct):
    if pct <= 0.70:
        return 'Category A'
    elif pct <= 0.90:
        return 'Category B'
    else:
        return 'Category C'

df['ABC_Category'] = df['Cum_Pct'].apply(assign_abc)

# Executive Summary Output
print("=== INVENTORY ANALYSIS SUMMARY ===")
print(f"Total Retail Sales Revenue: ${df['Total_Sales_Revenue'].sum():,.2f}")
print(f"Total COGS: ${df['Total_COGS'].sum():,.2f}")
print("\nProduct Count by ABC Category:")
print(df['ABC_Category'].value_counts())

=== INVENTORY ANALYSIS SUMMARY ===
Total Retail Sales Revenue: $483,645.80
Total COGS: $964,046.65

Product Count by ABC Category:
ABC_Category
Category C    68
Category A    18
Category B    18
Name: count, dtype: int64


Embed Tableau Dashboard Inline
=


In [2]:
from IPython.display import IFrame

tableau_url = "https://public.tableau.com/views/DailyChallengeInventoryAnalysis/WarmeHandsInc_-InventoryAnalysis?:showVizHome=no&:embed=true"

IFrame(src=tableau_url, width=1100, height=850)

<hr>

<h2>Executive Summary & Key Insights</h2>

<p><b>1. ABC Inventory Classification (Revenue Prioritization)</b></p>
<ul>
  <li><b>Category A Products (~17% of SKUs):</b> Drive <b>70% of total company revenue</b> (USD 338.5K+). These high-velocity items must maintain strict safety stock levels and zero-stockout targets to protect core cash flow.</li>
  <li><b>Category B Products (~17% of SKUs):</b> Contribute the next <b>20% of revenue</b> (USD 96.7K). These items require standard replenishment thresholds and quarterly demand reviews.</li>
  <li><b>Category C Products (~66% of SKUs):</b> Generate only the final <b>10% of total revenue</b> (USD 48.4K). These represent a long tail of inventory that should be streamlined to minimize holding and administrative costs.</li>
</ul>

<hr>

<p><b>2. Category Turnover Efficiency</b></p>
<ul>
  <li><b>Top Performer:</b> <b>Office & School</b> leads operational efficiency with a <b>0.88 turnover rate</b>, effectively balancing inventory investment with sales velocity.</li>
  <li><b>Underperformer:</b> <b>Jewelry</b> sits at a critically low turnover rate of <b>0.15</b> (well below the company average of ~0.70), indicating overstocked capital and slow-moving inventory.</li>
</ul>

<hr>

<p><b>3. Actionable Inventory Recommendations</b></p>
<ul>
  <li><b>Capital Clearance in Low-Turnover Items:</b> Products located in the bottom-right quadrant of the Risk Matrix (high inventory value tied up with turnover rates under 0.30) should be targeted for promotional bundling or discount clearance to free up working capital.</li>
  <li><b>Purchasing Re-alignment:</b> Shift future purchasing budgets toward Category A items within <b>Office & School</b> and <b>Toys & Edibles</b>, while reducing re-order quantities for low-velocity Category C items in the <b>Jewelry</b> department.</li>
</ul>